# Bubbleless Phase 1 (v0.1) — Colab T4 异构气泡检测与首尾重叠实验

**使用前请确认**：菜单栏 `代码执行程序 (Runtime)` -> `更改运行时类型 (Change runtime type)` -> 选择 **`T4 GPU`**。

本实验在真实 **NVIDIA T4 (15GB 显存 + 2核 CPU + 12.7GB 内存)** 上验证：
1. **三组件探针（IO / CPU / GPU）**：精准抓出视频生成流水线里的 GPU 空等气泡（Bubble Time）。
2. **首尾无损异步重叠（Phase 1）**：将读图解码（IO+CPU）与 H.264 视频编码落盘（CPU+IO）同 GPU 计算重叠，同时受内存水位保护。
3. **可视化泳道图 + Perfetto JSON 导出**：直观看到消除了哪些气泡、还剩哪些气泡留给第二阶段（PCIe 权重搬运）。


### Step 1：检查 T4 GPU 硬件环境并安装依赖


In [ ]:
!nvidia-smi
!lscpu | grep -E 'Model name|CPU\(s\):'
!free -h
!pip install -q diffusers transformers accelerate safetensors psutil pillow matplotlib


### Step 2：从 GitHub 拉取最新 Bubbleless 代码


In [ ]:
!rm -rf /content/bubbleless
!git clone --depth 1 https://github.com/cn7shi/bubbleless.git /content/bubbleless


### Step 3：运行 A/B 对比实验（串行 Baseline vs 异步双缓冲 Overlap）
默认连续跑 **5 个 16帧 AI 视频重绘任务**（开启 CPU<->GPU 模型卸载模拟消费级显存受限场景），并做 SHA-256 无损校验。


In [ ]:
%cd /content/bubbleless
!python3 run_phase1_exp.py --num-tasks 5 --steps 12 --work-dir /content/bubbleless_exp


### Step 4：直接在页面内查看【IO / CPU / GPU 气泡泳道对比图】并下载 Perfetto Trace


In [ ]:
from IPython.display import Image as IPyImage, display
import pathlib

trace_dir = pathlib.Path('/content/bubbleless_exp/traces')
display(IPyImage(filename=str(trace_dir / 'timeline_phase1_comparison.png')))

try:
    from google.colab import files
    print('正在触发下载 Perfetto 时间轴文件（可拖入 https://ui.perfetto.dev 放大查看细节）：')
    files.download(str(trace_dir / 'trace_phase1_serial.json'))
    files.download(str(trace_dir / 'trace_phase1_overlap.json'))
except Exception as e:
    print('Trace files saved at:', trace_dir)
